In [5]:
import  joblib
import numpy as np, pandas as pd

In [6]:
# Load test file
test_df = pd.read_excel("FinalTestDataset2025.xls")


print("Loaded test shape:", test_df.shape)
if 'ID' in test_df.columns:
    test_id = test_df['ID'].astype(str)
else:
    test_id = test_df.index.astype(str)

Loaded test shape: (133, 119)


In [7]:
#Find artifacts
selected_path = 'selected_features.pkl'
num_imputer_path = 'num_imputer.joblib'
model_path = 'svr_final_trained_on_all.joblib'


In [8]:
# Load artifacts

selected = joblib.load(selected_path)
num_imputer = joblib.load(num_imputer_path)
model = joblib.load(model_path)
print("Model loaded. Model expects n_features_in_ =", getattr(model, "n_features_in_", "unknown"))

Model loaded. Model expects n_features_in_ = 42


In [9]:

test_df = test_df.replace(999, np.nan)

missing_cols = [c for c in selected if c not in test_df.columns]
if missing_cols:
    print("Warning: these selected cols missing from test file (they will be filled with 0):", missing_cols)

X_test = pd.DataFrame(index=test_df.index)
for c in selected:
    if c in test_df.columns:
        X_test[c] = test_df[c]
    else:
        X_test[c] = 0.0  
if num_imputer is not None:
    try:
        X_test_imputed = pd.DataFrame(num_imputer.transform(X_test[selected]), columns=selected, index=X_test.index)
        X_test = X_test_imputed
    except Exception as e:
        print("Num imputer transform failed:", e)
        X_test = X_test.fillna(X_test.median())

else:
    X_test = X_test.fillna(X_test.median())

Num imputer transform failed: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- ChemoGrade
- HistologyType
- LNStatus
- PgR
- Proliferation
- ...



In [10]:
if hasattr(model, "n_features_in_"):
    if X_test.shape[1] != model.n_features_in_:
        raise ValueError(f"X_test has {X_test.shape[1]} features but model expects {model.n_features_in_}. Check selected_features.pkl and training artifacts.")
print("X_test prepared shape:", X_test.shape)

X_test prepared shape: (133, 42)


In [12]:
preds = model.predict(X_test)

In [13]:
# Save CSV
out = pd.DataFrame({'ID': test_id, 'RelapseFreeSurvival': preds})
out.to_csv('RFSPrediction.csv', index=False)
print("Saved RFSPrediction.csv shape:", out.shape)


Saved RFSPrediction.csv shape: (133, 2)
